In [1]:
import pandas as pd
from kaggle.api.kaggle_api_extended import KaggleApi

In [ ]:
'''import os
#pip install datasets si besoin
from datasets import load_dataset

def download_emotion_dataset(target_dir="data/raw"):
    os.makedirs(target_dir, exist_ok=True)
    print("Downloading dair-ai/emotion dataset from Hugging Face...")
    dataset = load_dataset("emotion")
    for split in dataset:
        out_path = os.path.join(target_dir, f"emotion_{split}.csv")
        dataset[split].to_csv(out_path, index=False)
        print(f"Saved {split} split to {out_path}")dataset'''

In [2]:
import importlib
import os
import sys

sys.path.insert(0, os.path.abspath(".."))
from src.data_cleaning.data import download_data, load_data, clean_data, balance_classes


/home/thomas_d/.pyenv/versions/MHSD/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
download_data()

In [3]:
df = load_data()
df_cleaned = clean_data(df)
df_balanced = balance_classes(df_cleaned, "label")

/home/thomas_d/code/Projet/mental-health-signal-detector/src/data_cleaning/data.py:54: DtypeWarning: Columns (0: Unnamed: 0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(_get_project_data_dir() / DATA_FILENAME)


In [4]:
df_balanced["label"].value_counts(), df_balanced.shape, df_balanced.isna().sum()

(label
 0.0    480409
 1.0    480409
 Name: count, dtype: int64,
 (960818, 2),
 title    0
 label    0
 dtype: int64)

In [5]:
# pip install nltk si besoin
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [6]:
_stop_words = set(stopwords.words('english'))
_lemmatizer = WordNetLemmatizer()

def preprocess_text(text, remove_stopwords=True, remove_punctuation=True, lemmatize=True):
    tokens = word_tokenize(text)
    tokens = [t.lower() for t in tokens]
    
    if remove_punctuation:
        tokens = [t for t in tokens if t.isalpha()]
    
    if remove_stopwords:
        tokens = [t for t in tokens if t not in _stop_words]
    
    if lemmatize:
        for pos in ('n', 'v', 'a'):
            tokens = [_lemmatizer.lemmatize(t, pos=pos) for t in tokens]
    
    return ' '.join(tokens)

In [7]:
df_balanced["clean_title"] = df_balanced["title"].apply(preprocess_text)

In [8]:
df_balanced[["title", "clean_title"]].head()

,title,clean_title
0,I haven't heard anything from my best/only fri...,hear anything friend day ever since ask could ...
1,"Gf advice needed plz Ok so, yesterday at the d...",gf advice need plz ok yesterday dance tell gir...
2,"Unpopular opinion: the word ""wholesome"" is gre...",unpopular opinion word wholesome great overuse...
3,I haven't left my house in a week On school ho...,leave house week school holiday go bed super l...
4,"I went to church today, for the lack of anythi...",go church today lack anything good really make...


In [9]:
df_balanced.drop(columns=["title"], inplace=True)

In [ ]:
#On sauvegarde en .csv
#df_balanced.to_csv("../../data/emotion_balanced.csv", index=False)

In [ ]:
#On sauvegarde la data en .pkl pour éviter de refaire le nettoyage à chaque fois
#df_balanced.to_pickle("../data/emotion_balanced.pkl")

In [48]:
# TODO: compute the BOW
from sklearn.feature_extraction.text import CountVectorizer
from scipy import sparse

# We create the output BOW, we can even reject directly the stop words and the punctuation, how magical?
vectorizer = CountVectorizer(max_features=10000,
                             stop_words='english',
                             #min_df=0.001, max_df=0.999
)
X = vectorizer.fit_transform(df_balanced["clean_title"])  # scipy sparse matrix
print(type(X), X.shape)  # should be (num_docs, num_features)

BOW = pd.DataFrame.sparse.from_spmatrix(
    X,
    index=df_balanced.index,
    columns=vectorizer.get_feature_names_out(),
)

<class 'scipy.sparse._csr.csr_matrix'> (960818, 10000)


In [15]:
#On vérifie si BOW est bien une sparse matrix
sparse.issparse(BOW)

True

In [21]:
df_balanced["clean_title"][899]

'depress worthless end life would smart thing ive ever do awful person ive burden many people year jump job job look something didnt hate find doesnt exist give opportunity jump ship new excite place exactly job worthless hour week go dr bill dad support sure lift debt lose everything ive take much ive give contribute nothing world ive consume dont want try get back life hat want die deserve die beyond hopeless anymore feel like tomorrow think finally time shouldve do year ago dont want run away start anew dont like way world work fuck cant accept happiness need fight something inside u people tell worldviews cant comprehend hate world love family dont want around lose guess make selfish lazy doesnt surprise piece shit like bill burr say people simply dont deserve im one'

In [49]:
y = df_balanced["label"]
X.shape, y.shape

((960818, 10000), (960818,))

In [50]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    test_size=0.2, random_state=42,
                                                    stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((768654, 10000), (192164, 10000), (768654,), (192164,))

In [ ]:
# TODO: Perform a logistic regression to predict whether a message is a spam or not
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)
log_reg.score(X_test, y_test)